# Cleaning 2.3 - Clean energy calculations

This notebook does the following:
    (1) Creates variables so that the dataset can be merged with the equipment calculators
    (2) Merges the dataset with the equipment calculators
    (3) Performs the energy calculations

## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [ ]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [ ]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

## (1) Prepare for merging with equipment calculators

In [ ]:
# Clean number variable

# If number is missing, set number to 1
equipment.loc[equipment["number"].isna(), "number"] = 1

# Make number integer
equipment["number"] = equipment["number"].astype(int)


In [ ]:
# Clean hours and days variables
# If hours is missing or "Unknown", set hours to 9

#Add mask for equipment that has these variables, loop over values in tuple for each value in equipment type
has_day_hours_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("fc", "heater", "bath", "cryostat", "microbio", "glassware", "it")))
#always_on = ("fridge", "freezer", "ult") #Included for completeness

equipment.loc[(has_day_hours_mask & (equipment["hours"].isna()) | (equipment["hours"] == "Unknown")), "hours"] = 9

# If days is missing or "Unknown", set days to 255
equipment.loc[(has_day_hours_mask & (equipment["days"].isna()) | (equipment["days"] == "Unknown")), "days"] = 255

# Make hours and days numeric
equipment["hours"] = pd.to_numeric(equipment["hours"], errors="raise")
equipment["days"] = pd.to_numeric(equipment["days"], errors="raise")

In [ ]:
# 1. Fix all data within its respective starting column

In [ ]:
# Clean FC variables

# Check unique values in hours_open
print("Unique values in hours_open before cleaning:")
print(equipment["hours_open"].unique())

# Fill missing/Unknown values for hours_open
equipment = fill_with_equipment_mode(
    equipment,
    value_col="hours_open",
    equipment_value="fc",
)

# Make hours_open numeric
equipment["hours_open"] = pd.to_numeric(equipment["hours_open"], errors="raise")

# Create "Type" variable which is "VAV" if "controller_type" is "Variable Air Volume (VAV)" or "Unknown" or missing, and "CAV" if "controller_type" is "Constant Air Volume (CAV)" ONLY if equipment is "fc"
equipment["Type"] = np.where( #TODO make this controller type to stack later
    (equipment["controller_type"].isin(["Variable Air Volume (VAV)", "Unknown"]) | equipment["controller_type"].isna()) 
     & (equipment["equipment"] == "fc"),
    "VAV",
    np.where((equipment["controller_type"] == "Constant Air Volume (CAV)") & (equipment["equipment"] == "fc"),
             "CAV",
             None)
)

#TODO convert width to meters and numeric, check if in 0.5-3

#TODO Input Andy advice for missing face velocity
#TODO Raised: Replace those saying they have nothing in the fc nothing inside comments with yes, since having no items is the same as them being raised (flag for martin)
# print(equipment["Type"].unique())

In [ ]:
# Clean Fridge Variables

##TODO extract just numbers 250-425 -> 250-450
print(equipment["size_fridge"].unique())

#To extract just numbers, take piece before (~, drop )
#replace swaps out changed range value
equipment["size_fridge_2"] = equipment["size_fridge"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"250-425L": "250-450L"})
# Strings which contain fan are classified as such, convection likewise, and then remaining values should remain the same (nan/Unknown)
equipment["size_fridge_1"] = np.where(
    equipment["size_fridge"].str.contains("fan", na=False), "Fan",
    np.where(equipment["size_fridge"].str.contains("convection", na=False), "Convection",
             equipment["size_fridge"])

)
#TODO Below is potentially "clearer" but definitely longer. Do I delete it?
# equipment["size_fridge_2"] = equipment["size_fridge"].replace({'Under bench fan assisted (~100-160L)': "100-160L",  
#                                                                'Under bench convection (~100-160L)': "100-160L",
#                                                                'Tall single door fan assisted (~250-425L)': "250-450L",
#                                                                'Tall single door fan assisted (~500-700L)': "500-700L",
#                                                                'Tall single door convection (~250-425L)': "250-450L",
#                                                                'Under bench fan assisted (~250-425L)': "250-450L",
#                                                                'Tall double door fan assisted (~1200-1500L)': "1200-1500L",
#                                                                'Tall single door convection (~100-160L)': "100-160L"})

# equipment["size_fridge_1"] = np.where(
#     equipment["size_fridge"].isin(["Under bench fan assisted (~100-160L)", "Tall single door fan assisted (~250-425L)",
#                                    "Tall single door fan assisted (~500-700L)", "Under bench fan assisted (~250-425L)",
#                                    "Tall double door fan assisted (~1200-1500L)"]),
#                                    "Fan",
#                                    np.where(equipment["size_fridge"].isin(["Under bench convection (~100-160L)",
#                                                                            "Tall single door convection (~250-425L)",
#                                                                            "Tall single door convection (~100-160L)"]),
#                                                                            "Convection",
#                                                                            equipment["size_fridge"])

# )
#TODO ensure fridge has no door_opening option (check to make sure it matches spreadsheet penalty)
#TODO door opening is now in minutes!

In [ ]:
# Clean Freezer variables

#TODO Size and Type are now split out of size_freezer
#80-130L -> 90-130L, 190-38L -> 190-380L
#To be added to Size
equipment["size_freezer_2"] = equipment["size_freezer"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"80-130L": "90-130L",
                                                                                                                  "190-38L": "190-380L"}) #Take piece before (~, drop ) TODO ugly
#To be added to type
equipment["size_freezer_1"] = np.where(
    equipment["size_freezer"].str.contains("Under bench", na=False), "Under Bench",
    np.where(equipment["size_freezer"].str.contains("Chest", na=False), "Chest",
             np.where(equipment["size_freezer"].str.contains("Tall", na=False), "Upright",
                      equipment["size_freezer"]))
)

#TODO Which does Daphne prefer for clarity?
# equipment["size_freezer_2"] = equipment["size_freezer"].replace({"Under bench (~80-130L)": "90-130L",
#                                                                  "Tall and slim single door (~190-38L)": "190-380L",
#                                                                  "Chest (~190-380L)": "190-380L",
#                                                                  "Tall and wide single door (~390-525L)": "390-525L",
#                                                                  "Chest (~390-525L)": "390-525L"})
# equipment["size_freezer_1"] = equipment["size_freezer"].replace({"Under bench (~80-130L)": "Under Bench",
#                                                                  "Tall and slim single door (~190-38L)": "Upright",
#                                                                  "Chest (~190-380L)": "Chest",
#                                                                  "Tall and wide single door (~390-525L)": "Upright",
#                                                                  "Chest (~390-525L)": "Chest"})
#TODO No refrigerant variable
#TODO Icing is now a No or Yes question, so "a little bit" must become (no)
#Drawer type has switched from Yes+Plastic/NoDrawers/No+Wire to Plastic/Wire
print(equipment["size_freezer_1"].unique())
print(equipment["size_freezer_2"].unique())

In [ ]:
# Clean ULT variables
#TODO match all variable names to Isobel table
#TODO two type questions, one for engine and one for refridgerent + age. First type can limit the second type. Second type is classified as Age by Bell
#ult_type splits into type, age takes the refrigerant
print(equipment["ult_type"].unique())
#Extract engine type, to be put in Type column
equipment["ult_type_1"] = np.where(
    equipment["ult_type"].str.contains("Dual compressor", na=False), "Dual compressors",
    np.where(equipment["ult_type"].str.contains("Cascade", na=False), "Cascade compressors",
             np.where(equipment["ult_type"].str.contains("Stirling Engine", na=False), "Stirling",
                      np.where(equipment["ult_type"].str.contains("Unknown", na=False), "Unknown", #TODO no longer a valid option
                               equipment["ult_type"])))
)
#Extract age and refrigerant type to put into the age column
#TODO no ability to mark unknown 
equipment["ult_type_2"] = np.where(
    (equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HC",
    np.where((equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HC",
             np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HFC",
                      np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HFC",
                               np.where((equipment["ult_type"].str.contains("Unknown", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "Unknown", equipment["ult_type"])))) #TODO Unknown value not supported
)

#TODO Temp 78, 81, 82 -> 80 / Only options are 70, 75, 80
#TODO Return here

#TODO Seals dirty -> Clean, only Damaged/Not Damaged now (I presume this means that the warnings are eliminated)
# Policy on how to replace categories that are the same but overlap?
# print(equipment["seals"].unique())
equipment["seals"] = equipment["seals"].replace({"Dirty or lightly iced": "No, seals are intact"})

# >= 150/Little spacing -> Good/Little
# print(equipment["spacing"].unique())
equipment["spacing"] = equipment["spacing"].replace({"There's little space around the unit and we store items on top of the unit.": "Little / no spacing",
                                                     "≥150mm gap at the back and sides of the unit, with no items stored on top.": "Good spacing around unit (>= 150 mm on all sides)"})

#TODO Filter little dirty -> Clear
# print(equipment["filter"].unique())
# equipment["filter"] = equipment["filter"].replace({"It's a little dirty": "No, filters are clear"})


In [ ]:
# Clean Glassware Variables
#TODO match all variable names to Isobel table

#tech is now age
# print(equipment["tech"].unique())
equipment["Age"] = equipment["tech"].replace({"The chamber is insulated, it's warm to the touch when on (modern technology)" : "Newer",
                                                                 "The chamber has no insulation, it's hot to touch when on (older technology)" : "Older"})

#TODO Fan has no I don't know option
#TODO 51 -> 50, 65 -> 60, 80, 81 -> 75


In [ ]:
# Clean MSC Variables

#TODO match all variable names to Isobel table
#TODO No I don't know for age


In [ ]:
# Clean Incubator Variables
#TODO match all variable names to Isobel table
#No <150L, Add to 150-170L
# print(equipment["capacity_incubator"].unique())
equipment["capacity_incubator"] = equipment["capacity_incubator"].replace({"<150L": "150-170L",
                                                                           "~220-260L": "220-260L"})

#><20 -> Newer/Older
# print(equipment["age_incubator"].unique())
# equipment["age_incubator"] = equipment["age_incubator"].replace({"≤ 20 years": "Newer", "> 20 years": "Older"})

In [ ]:
# Clean water bath variables

# Fill missing/Unknown values for capacity_bath, heating, temp_bath, lid
for col in ["capacity_bath", "heating", "temp_bath", "lid"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="bath",
    )

# Create "Size" variable which is the same as "capacity_bath" except combining "10-12L" and "6-10L" into "10L"
equipment["Size"] = equipment["capacity_bath"].replace({"10-12L": "10L", "6-10L": "10L"})
#TODO No less than 5L
#TODO Should 10-12 be 12-14 to keep the ranges similar? Check Bel code

# Set Set_Temp only for water bath rows
bath_mask = equipment["equipment"] == "bath"
#TODO 85 -> 90, 42, 40, 49, 46, 22(?) -> 37, 51 -> ?, 56, 58 -> 65 (Data -> SPARKHub val)
equipment.loc[bath_mask, "Set_Temp"] = equipment.loc[bath_mask, "temp_bath"].apply(
    lambda t: assign_set_temp(t, [37, 65, 90])
)

In [ ]:
# Clean cryostat variables

# Fill missing/Unknown values for temp_cryostat and sleep_mode
for col in ["temp_cryostat", "sleep_mode"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="cryostat",
    )

# Print unique values in sleep_mode
print(equipment["sleep_mode"].unique())

# Fill in "Type" variable with "Energy Saving Mode Available"/"No Energy Saving Mode Available" based on sleep_mode
equipment["Type"] = np.where(
    (equipment["sleep_mode"].isin([
        "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode",
        "Yes, but it's not used",
        "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"
        ]) | equipment["sleep_mode"].isna())    
        & (equipment["equipment"] == "cryostat"),
    "Energy Saving Mode Available",
    np.where((equipment["sleep_mode"] == "No, it does not")
             & (equipment["equipment"] == "cryostat"),
             "No Energy Saving Mode Available",
             equipment["Type"])
)

# Set Set_Temp only for cryostat rows
cryostat_mask = equipment["equipment"] == "cryostat"
equipment.loc[cryostat_mask, "Set_Temp"] = equipment.loc[cryostat_mask, "temp_cryostat"].apply(
    lambda t: assign_set_temp(t, [-25, -20, -18])
)
#TODO Assign possible options function?
print(equipment["Type"].unique())

In [ ]:
# Clean it variables

# Check unique values of it_type
print(equipment["it_type"].unique())

# Fill missing/Unknown values for it_type
equipment = fill_with_equipment_mode(
    equipment,
    value_col="it_type",
    equipment_value="it",
)

#TODO monitor variable split for size and brightness

In [ ]:
# Map all variables to their new names
print(equipment["equipment"].unique())

# 3. Create columns to match
## Function which shifts all the relevant data to the same column name that Isobel used.
## Changes regarding correct exact entries to be made afterwards
## Each section has a function which ensures only the relevant equipment have their values from these columns selected
## Other equipment gets None as the value


# Type = [controller_type, first halt size_fridge, first halt size_freezer, first half of ult_type, capacity_glassware, sleep_mode, it_type]
def equipment_type_func(row):
    if row["equipment"] == "fc":
        return row["controller_type"]
    elif row["equipment"] == "fridge":
        return row["size_fridge_1"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_1"]
    elif row["equipment"] == "ult":
        return row["ult_type"]
    elif row["equipment"] == "glassware":
        return row["capacity_glassware"]
    elif row["equipment"] == "cryostat":
        return row["sleep_mode"]
    elif row["equipment"] == "it":
        return row["it_type"]
    else:
        return None
equipment["Type"] = equipment.apply(lambda row: equipment_type_func(row), axis=1)

# Work_Surface_Width = [sash_width, width]
def equipment_width_func(row):
    if row["equipment"] == "fc":
        return row["sash_width"]
    elif row["equipment"] == "microbio":
        return row["width"]
    else:
        return None
equipment["Work_Surface_Width"] = equipment.apply(lambda row: equipment_width_func(row), axis=1)
#TODO Convert to numeric

# Size = [second half size_fridge, second half size_freezer, second half size_ult, capacity_bath, capacity_incubator, first half monitor]
def equipment_size_func(row):
    if row["equipment"] == "fridge":
        return row["size_fridge_2"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_2"]
    elif row["equipment"] == "ult":
        return row["ult_type"]
    elif row["equipment"] == "bath":
        return row["capacity_bath"]
    elif row["equipment"] == "incubator":
        return row["capacity_incubator"]
    elif row["equipment"] == "it":
        return row["monitor"]
    else:
        return None
equipment["Size"] = equipment.apply(lambda row: equipment_size_func(row), axis=1)

# N_blocks = [blocks]
equipment["N_blocks"] = equipment["blocks"]
#TODO convert to numeric

# Age = [refrigerant (combined with age value), secold half of ult_type (with age), tech, age_microbio, age_incubator]
def equipment_age_func(row):
    if row["equipment"] == "fridge":
        return row["refrigerant"]
    elif row["equipment"] == "ult":
        return row["ult_type"]
    elif row["equipment"] == "cryostat":
        return row["tech"]
    elif row["equipment"] == "microbio":
        return row["age_microbio"]
    elif row["equipment"] == "incubator":
        return row["age_incubator"]
    else:
        return None
equipment["Age"] = equipment.apply(lambda row: equipment_age_func(row), axis=1)

# Set_Temp = [temp_freezer, temp_ult, temp_glassware, temp_cryostat, temp_bath, temp_heater]
def equipment_temp_func(row):
    if row["equipment"] == "freezer":
        return row["temp_freezer"]
    elif row["equipment"] == "ult":
        return row["temp_ult"]
    elif row["equipment"] == "glassware":
        return row["temp_glassware"]
    elif row["equipment"] == "cryostat":
        return row["temp_cryostat"]
    elif row["equipment"] == "bath":
        return row["temp_bath"]
    elif row["equipment"] == "heater":
        return row["temp_heater"]
    else:
        return None
equipment["Set_Temp"] = equipment.apply(lambda row: equipment_temp_func(row), axis=1)

# Brightness = [second half monitor]
equipment["Brightness"] = equipment["monitor"]

# Penalty indicators = [icing, drawers, seals, spacing, filter, fan, ducting, heating, lid]

# fc_specific = [face_velocity, lifted, surface, ] TODO look at code for this information

## Clean up and save processed data

In [ ]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_3.csv", index =False)